# Mframapa AI - Universal African Air Quality Model
## Phase 4: Advanced Training (Hybrid GPU/CPU)

**Instructions:**
1. **Runtime**: Ensure you are using **T4 GPU**.
2. **Install**: Run the first cell to install RAPIDS.
3. **Run**: The script will try to load data to GPU memory. If it's too big, it automatically falls back to System RAM while still training on GPU.

In [ ]:
# Install RAPIDS (Optional check)
!pip install cudf-cu12 --extra-index-url=https://pypi.nvidia.com
print("Libraries Checked.")

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import xgboost as xgb
import os
import numpy as np_cpu
import pandas as pd_cpu
from sklearn.model_selection import RandomizedSearchCV, GroupKFold
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import uniform, randint
import joblib
import gc # IMPORT GC

# Try importing RAPIDS; if fails, we run in CPU-only mode for data loading
try:
    import cudf as pd_gpu
    import cupy as np_gpu
    HAS_RAPIDS = True
    print("✅ RAPIDS (GPU Acceleration) is available.")
except ImportError:
    HAS_RAPIDS = False
    print("⚠️ RAPIDS not found. Will use System RAM for data loading.")

### 1. Hybrid Data Loader
Tries to fit data in GPU VRAM (Fastest). If OOM, switches to System RAM (Standard).

In [ ]:
def load_dataset_hybrid():
    paths = [
        "/content/sample_data/training_dataset.csv",
        "/content/training_dataset.csv",
        "training_dataset.csv",
        "../data/training_dataset.csv",
        "backend/data/training_dataset.csv"
    ]

    needed_cols = [
        'pm25', 'location', 'datetime', 'date_str', 'lat', 'lon',
        'sat_no2', 'sat_aod', 'sat_aot',
        'sat_pblh', 'sat_humidity', 'sat_rh',
        'sat_wind_u', 'sat_wind_v', 'pop_density'
    ]
    
    dtypes = {
        'pm25': 'float32',
        'sat_no2': 'float32', 'sat_aod': 'float32', 'sat_aot': 'float32',
        'sat_pblh': 'float32', 'sat_humidity': 'float32', 'sat_rh': 'float32',
        'sat_wind_u': 'float32', 'sat_wind_v': 'float32', 'pop_density': 'float32',
        'lat': 'float32', 'lon': 'float32'
    }

    for p in paths:
        if os.path.exists(p):
            print(f"Found dataset at: {p}")
            
            # Get available columns first using CPU pandas (cheap)
            try:
                all_cols = pd_cpu.read_csv(p, nrows=0).columns.tolist()
                use_cols = [c for c in needed_cols if c in all_cols]
                print(f"Target Columns: {len(use_cols)}")
            except Exception as e:
                print(f"Error reading header of {p}: {e}")
                continue

            # --- ATTEMPT 1: GPU LOAD ---
            if HAS_RAPIDS:
                try:
                    print("🚀 Attempting GPU VRAM Load...")
                    gdf = pd_gpu.read_csv(p, usecols=use_cols, dtype=dtypes)
                    print(f"✅ Success! Loaded {len(gdf)} rows into VRAM.")
                    return gdf, "GPU"
                except Exception as e:
                    print(f"⚠️ GPU OOM or Error ({e}). Switching to System RAM...")
            
            # --- ATTEMPT 2: CPU LOAD (Fallback) ---
            try:
                print("💾 Loading into System RAM (Chunked)...")
                chunk_list = []
                for chunk in pd_cpu.read_csv(p, usecols=use_cols, dtype=dtypes, chunksize=50000):
                    chunk_list.append(chunk)
                df = pd_cpu.concat(chunk_list, ignore_index=True)
                print(f"✅ Success! Loaded {len(df)} rows into System RAM.")
                return df, "CPU"
            except Exception as e:
                print(f"❌ CPU Load Failed: {e}")
                continue

    raise FileNotFoundError("CRITICAL: Could not find 'training_dataset.csv'.")

df, MODE = load_dataset_hybrid()

### 2. Hybrid Feature Engineering
Automatically uses CuPy (GPU) or NumPy (CPU) based on where the data is.

In [ ]:
def engineer_features(df, mode):
    print(f"Engineering features using {mode} logic...")
    
    # Polyglot logic
    if mode == "GPU":
        pd = pd_gpu
        np = np_gpu
        # GPU datetime
        if 'datetime' in df.columns:
            df['dt'] = pd.to_datetime(df['datetime'], utc=True)
        elif 'date_str' in df.columns:
            df['dt'] = pd.to_datetime(df['date_str'])
    else:
        pd = pd_cpu
        np = np_cpu
        # CPU datetime
        if 'datetime' in df.columns:
            df['dt'] = pd.to_datetime(df['datetime'], utc=True)
        elif 'date_str' in df.columns:
            df['dt'] = pd.to_datetime(df['date_str'])
    
    if 'dt' not in df.columns:
        return df

    df['month'] = df['dt'].dt.month
    # Math works the same for both libraries if we aliased 'np' correctly
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    return df

df = engineer_features(df, MODE)

### 3. Training (GPU Enabled)
Even if data is on CPU, XGBoost will use the GPU.

In [ ]:
potential_features = [
    'sat_no2', 'sat_aod', 'sat_aot',
    'sat_pblh', 'sat_humidity', 'sat_rh',
    'sat_wind_u', 'sat_wind_v', 'pop_density',
    'month_sin', 'month_cos',
    'lat', 'lon'
]

features = [f for f in potential_features if f in df.columns]
target = 'pm25'

# Cleanup (Polyglot)
if MODE == "GPU":
    df = df.replace([np_gpu.inf, -np_gpu.inf], np_gpu.nan)
else:
    df = df.replace([np_cpu.inf, -np_cpu.inf], np_cpu.nan)

clean_subset = features + [target]
if 'location' in df.columns:
    clean_subset.append('location')

df = df.dropna(subset=clean_subset)

# Enforce float32
for col in features:
    df[col] = df[col].astype('float32')
df[target] = df[target].astype('float32')

X = df[features]
y = df[target]

# Cross-Validation setup
if 'location' in df.columns:
    if MODE == "GPU":
        groups = df['location'].to_numpy() # Force to CPU for GroupKFold
    else:
        groups = df['location']
    cv = GroupKFold(n_splits=5)
else:
    groups = None
    cv = 5

print(f"Final Rows: {len(X)}")

# CRITICAL: Garbage Collect to prevent Session Crash
# We don't need 'df' anymore, just X, y, and groups
print("🧹 Cleaning up raw dataframe to save RAM...")
del df
gc.collect()
print("✅ Memory Freed.")

param_dist = {
    'max_depth': randint(3, 10),
    'learning_rate': uniform(0.01, 0.3),
    'n_estimators': randint(100, 500),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4)
}

print("🚀 Starting XGBoost on GPU... (device='cuda')")
# NATIVE GPU SUPPORT
xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror', 
    n_jobs=-1, 
    random_state=42,
    tree_method='hist',
    device='cuda'  # <--- MAGIC LINE: Uses GPU for calculation regardless of input
)

search = RandomizedSearchCV(
    xgb_model, 
    param_distributions=param_dist, 
    n_iter=100, 
    cv=cv, 
    scoring='neg_root_mean_squared_error',
    verbose=1,
    random_state=42
)

if groups is not None:
    search.fit(X, y, groups=groups)
else:
    search.fit(X, y)

In [ ]:
# Save
best_model = search.best_estimator_
try:
    best_model.save_model("universal_african_model.json")
    print("Model Saved (JSON)")
except:
    joblib.dump(best_model, "universal_african_model.pkl")
    print("Model Saved (Pickle)")